# SDP-Meta Lab 1: Deploying and Onboarding new Pipelines

This guide walks you through deploying sdp-meta pipelines using the Declarative Automation Bundle (DABs) workflow in this repository. By the end you will have a running bronze+silver Lakeflow Declarative Pipeline ingesting CSV data via Autoloader. This guide will take you through deploying SDP-Meta from a Databricks Workspace.



## Prerequisites

| Requirement | Details |
| --- | --- |
| Workspace access | Unity Catalog enabled. You will need access to a Catalog and Schema in which you can create tables and volumes. You will need permission to create Jobs and Pipelines within your Workspace. You will need permission to start and run Serverless environments.
| Git repo access | Read access to https://github.com/rearc-data/sdp_meta_lab and the ability to create Git Folders from this repo |

In [0]:
import json, os
import re
from databricks.sdk import WorkspaceClient

def getAuthorName() -> str:
    w = WorkspaceClient()
    username = w.current_user.me().user_name

    if not username:
        raise ValueError("Username is null or empty. Ensure that you are running this notebook from a Databricks workspace.")

    # Sanitize username: take the local part before '@', lowercase,
    # and replace non-alphanumeric/underscore characters with underscores
    # so it is valid as part of a Unity Catalog schema name.
    username = username.strip().lower().split("@")[0]
    username = re.sub(r'[^a-z0-9_]', '_', username)
    username = re.sub(r'_+', '_', username).strip('_')

    return username

user_name = getAuthorName()
print(f"Current User: {user_name}")

sdp_meta_catalog = dbutils.widgets.get('LAB_CATALOG')
if not sdp_meta_catalog:
    raise Exception("Widget 'LAB_CATALOG' is empty. Please provide a value.")

sdp_meta_schema = dbutils.widgets.get('LAB_SCHEMA') or f"sdp_meta_{user_name}"
if not sdp_meta_schema:
    raise Exception("Widget 'LAB_SCHEMA' is empty. Please provide a value.")

dbutils.widgets.remove("LAB_SCHEMA")
dbutils.widgets.text("LAB_SCHEMA", sdp_meta_schema, "UC Schema")

print(f"This lab will use {sdp_meta_catalog}.{sdp_meta_schema} as the target schema")

## 1. Create a Git Folder from this repo

If you have already created a Git folder you can skip this section and continue with the guide.

A [Git Folder](https://docs.databricks.com/en/repos/index.html) syncs this repository into your workspace so you can run notebooks, edit configs, and deploy bundles directly.

### Steps

1. In the Databricks sidebar, click **Workspace**.
2. Navigate to your user directory (or any folder where you have write access).
3. Click the kebab menu (**⋮**) → **Create** → **Git Folder**.
4. In the dialog, enter:
   * **Git repository URL**: `https://github.com/rearc-data/sdp_meta_lab`
   * **Git provider**: `GitHub`
   * **Git folder name**: leave the default (`sdp_meta_lab`) or choose a custom name
   * **Branch**: select the branch you want to work from (e.g. `main`)
5. Click **Create Git Folder**.

Databricks will clone the repository into your workspace. Once complete, you will see all project files — including `databricks.yml`, `resources/`, `conf/`, and the lab notebooks


### References

* [Git integration with Databricks Git Folders](https://docs.databricks.com/en/repos/index.html)
* [Clone a remote Git repository](https://docs.databricks.com/en/repos/git-operations-with-repos.html#clone-a-remote-git-repository)
* [Best practices for Git Folders](https://docs.databricks.com/en/repos/repos-best-practices.html)

## 2. Understand the Bundle Variables

All tuneable parameters live in `resources/variables.yml`. The most important ones for a first deploy:

| Variable | Purpose | Default |
| --- | --- | --- |
| `uc_catalog_name` | Target Unity Catalog catalog | `sdp_meta_lab` |
| `sdp_meta_schema` | Schema for dataflowspec metadata tables | `sdp_meta_20260826` |
| `bronze_target_schema` | Schema where bronze tables land | `sdp_meta_20260826` |
| `silver_target_schema` | Schema where silver tables land | `sdp_meta_20260826` |
| `layer` | Which layers to onboard: `bronze`, `silver`, or `bronze_silver` | `bronze_silver` |
| `sdp_meta_dependency` | Wheel path or PyPI coordinate for the sdp-meta library | Volume wheel path |
| `serverless` | Run pipelines on serverless compute | `true` |
| `conf_volume` | UC Volume for staged configuration files within the target Catalog and Schema | `conf` |



### 3. Set the Deployment Variable Overrides

We will now set up the deployment variables for your environment.


- _Run the cell below to edit `.databricks/bundle/dev/variable-overrides.json`_


Here is a description of the parameters we are setting up:


```json
  "uc_catalog_name": "sdp_meta_lab",
  "sdp_meta_schema": "sdp_meta_<YOUR USERNAME>"
```
- The target Catalog and Schema location where you lab tables will deploy.


```json  
  "wheel_source":  "/Volumes/sdp_meta_lab/sdp_meta_20260826/conf/wheels/",
  "sdp_meta_dependency": "/Volumes/sdp_meta_lab/sdp_meta_20260826/conf/wheels/databricks_labs_sdp_meta-0.1.0-py3-none-any.whl",
```
- You can change these settings to load the sdp_meta wheel file to a new location.


```json  
  "overwrite_dataflowspecs": "True"
```
- overwrite_dataflowspecs should be True for new deployments to create the tables. After the first deploy you can set this to False and updates will be merged instead of overwriting the full table.


Note - In a production scenario, you would add the variables to the `databricks.yml` under `targets.<environment>.variables`.

In [0]:
import json, os
import re
from databricks.sdk import WorkspaceClient

print(f"Target Schema: {sdp_meta_catalog}.{sdp_meta_schema}")

if not user_name or not sdp_meta_catalog or not sdp_meta_schema:
  raise Exception ("Please run cell 3 above.")

# define variable overrides for deploying the lab

variable_overrides = {
  "author": user_name,
  "uc_catalog_name": sdp_meta_catalog,
  "sdp_meta_schema": sdp_meta_schema,
  "onboarding_file_name": "onboarding.json",
  "conf_volume": "sdp_meta",
  "overwrite_dataflowspecs": "True",
  "bronze_target_schema": sdp_meta_schema,
  "silver_target_schema": sdp_meta_schema
}


# overwrites the variable-overrides file with the defined variables

overrides_path = "../.databricks/bundle/dev/variable-overrides.json"
os.makedirs(os.path.dirname(overrides_path), exist_ok=True)

with open(overrides_path, "w") as f:
  json.dump(variable_overrides, f, indent=2)

print(f"Wrote variable overrides to {overrides_path}")
print(json.dumps(variable_overrides, indent=2))


## 4. Prepare the Catalog and Schema

Before deploying, ensure the target catalog and schema exist:

```sql
CREATE CATALOG IF NOT EXISTS sdp_meta_lab;
CREATE SCHEMA IF NOT EXISTS sdp_meta_lab.sdp_meta_${USERNAME};
CREATE VOLUME IF NOT EXISTS
sdp_meta_lab.sdp_meta_${USERNAME}.sdp_meta
```

The onboarding job will create the UC Volume (`conf`) automatically if it does not exist.

In [0]:
try:
    spark.sql(f'DESCRIBE CATALOG {variable_overrides["uc_catalog_name"]}')
except Exception as e:
    raise RuntimeError(
        f'Catalog \'{variable_overrides["uc_catalog_name"]}\' does not exist. '
        f'Please specify a UC Catalog in the LAB_CATALOG widget.'
    ) from e

try:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {variable_overrides["uc_catalog_name"]}.{variable_overrides["sdp_meta_schema"]}');
except Exception as e:
    raise RuntimeError(
        f'Failed to create UC Schema {variable_overrides["uc_catalog_name"]}.{variable_overrides["sdp_meta_schema"]}'
        f'Please specify a UC Schema in the LAB_SCHEMA widget and confirm that you have permissions to `CREATE VOLUME`, `CREATE TABLE`, and `CREATE FUNCTION` on this schema.'
    ) from e


spark.sql(f'CREATE VOLUME IF NOT EXISTS {variable_overrides["uc_catalog_name"]}.{variable_overrides["sdp_meta_schema"]}.{variable_overrides["conf_volume"]}');

## 5. Stage the Test Data

We copy our sample data to a volume location so that it will be accessible to Serverless workflows

In [0]:
#run this cell to upload the lab data files to the volume location

uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]
conf_volume = variable_overrides["conf_volume"]

data_dir=f"/Volumes/{uc_catalog_name}/{sdp_meta_schema}/{conf_volume}/data"
src_data_dir = f"file:{os.path.abspath('../data')}"

#ensure path exists
dbutils.fs.mkdirs(data_dir)

#clear any existing data files
dbutils.fs.rm(data_dir, recurse=True)

#upload all data files
dbutils.fs.cp(src_data_dir, data_dir, recurse=True) 

## 6. Update the Onboarding configuration files

The Onboarding configuration files are a set of files that are used to define the pipelines in your sdp-meta bundle.

The Onboarding job will read these files and load the Onboarding Spec tables.

DLT-Meta supports both JSON and YAML format. This lab will use the JSON format.

The relevant files are found in the `conf` folder of this project.

- `onboarding.json` - the root configuration file that will define the deployed pipelines.

- dqe files
  - host_status_updates_bronze_dqe.json
  - host_status_updates_siver_dqe.json

  Defines the Expectations that will be run on the records before they are loaded into the destination tables.

  The following Expectation types are supported:
  - "expect" - Logs any failures (records are still loaded)
  - "expect_or_drop" - Logs any failures and records are dropped.
  - "expect_or_fail" - Fails the pipeline
  - "expect_or_quarantine" - Failing records are sent to a  quarantine table

- silver transformation files
  - host_status_updates_silver_transformations

  Defines the transformations that will be run when loading the silver table from bronze.


- Schemas
Optional files that define the schemas used by the Cloudfiles configuration of the bronze ingest section of the pipeline.

Examine the files and get familiar with the contents and how they are used to define the pipelines in the sdp-meta bundle.

Then, update the onboarding.json file to target tables inside your lab environment Catalog and Schema.

Update bronze_database_dev and silver_database_dev to reference the correct target schema.





In [0]:

import json

sdp_meta_schema = variable_overrides["sdp_meta_schema"]
uc_catalog_name = variable_overrides["uc_catalog_name"]
conf_volume = variable_overrides["conf_volume"]
data_dir=f"/Volumes/{uc_catalog_name}/{sdp_meta_schema}/{conf_volume}/data"

template_path = "../conf/templates/onboarding.template.json"
onboarding_path = "../conf/onboarding.json"

with open(template_path, "r") as f:
    onboarding = json.load(f)

if not isinstance(onboarding, list):
    onboarding = [onboarding]

for entry in onboarding:
    entry["bronze_database_dev"] = sdp_meta_schema
    entry["silver_database_dev"] = sdp_meta_schema
    entry["bronze_catalog_dev"] = uc_catalog_name
    entry["silver_catalog_dev"] = uc_catalog_name

    if entry["data_flow_id"] == "host_status_updates_csv_volume":
        entry["source_details"]["source_path_dev"] = data_dir

with open(onboarding_path, "w") as f:
    json.dump(onboarding, f, indent=2)

print(f"Updated onboarding.json: bronze_database_dev and silver_database_dev set to '{sdp_meta_schema}'")
print(json.dumps(onboarding, indent=2))



## 7. Validate the Bundle

To validate the bundle open the Deployments section from the Workspace interface.

Any validation error in the bundle will be displayed on this tab.

Alternatively the bundle can be validated by running `databricks bundle validate` using the Web Terminal.

## 8. Deploy the Bundle

To deploy the bundle open the Deployments section from the Workspace interface.

Confirm the target is `dev` and then hit the Deploy button.

The following resources should be deployed:
```
Created jobs.onboarding
Created jobs.pipelines
Created pipelines.bronze_silver
Files: 0 uploaded, 0 deleted
Resources: 3 created, 0 changed, 0 deleted, 0 unchanged
```

The bundle can also be deployed from the Web Terminal using the command `databricks bundle deploy --target dev`

## 9. Run Onboarding


Run the SDP-Meta onboarding job. The purpose of this job is to load the configuration files into the dataflowspecs bronze and silver tables.

Once the job completes, validate the job succeeded by verifying the dataflow spec tables have been created.

Examine the entries to verify how the configuration files are loaded into the Delta tables.

Normally, you will never need to edit the dataflow spec tables directly. Instead update the configuration files, deploy the bundle, then run the Onboarding job to update the dataflow spec tables with the changes. These changes will automatically be picked up the next time the pipelines are invoked.



In [0]:
#validate the onboarding specs tables

uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

bronze_dataflowspec_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.bronze_dataflowspec');

silver_dataflowspec_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.silver_dataflowspec');

display(bronze_dataflowspec_df)

display(silver_dataflowspec_df)

assert bronze_dataflowspec_df.filter("dataFlowId = 'host_status_updates_csv_volume'").count() > 0, "bronze_dataflowspec_df missing expected pipeline 'host_status_updates_csv_volume'"
assert silver_dataflowspec_df.filter("dataFlowId = 'host_status_updates_csv_volume'").count() > 0, "silver_dataflowspec_df missing expected pipeline 'host_status_updates_csv_volume'"

print("Onboarding Spec validation passed.")

## 10. Run the Pipeline

Run the pipeline `sdp-meta-lab - run pipelines`.

Verify that the pipeline completes successfully.

## 11. Verify Results

Once the pipeline completes, query the output tables:

```sql
-- Bronze table (raw ingestion via Autoloader)
SELECT * FROM sdp_meta_lab.sdp_meta_${USERNAME}.host_status_updates_csv_bronze;

-- Silver table (cleaned and transformed)
SELECT * FROM sdp_meta_lab.sdp_meta_${USERNAME}.host_status_updates_csv_silver;
```

Expected results:
* Bronze contains all rows from `sample.csv` with Autoloader metadata columns
* Silver contains filtered, type-cast rows with a `processed_at` timestamp
* Rows failing DQE `expect_or_drop` rules (null `system_id` or `start_date`) are excluded
* Rows failing `expect_or_fail` rules (invalid `status`) cause the pipeline to error

In [0]:
#validate the bronze and silver tables

uc_catalog_name = variable_overrides["uc_catalog_name"]
sdp_meta_schema = variable_overrides["sdp_meta_schema"]

bronze_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_csv_bronze');

silver_df = spark.sql(f'SELECT * FROM {uc_catalog_name}.{sdp_meta_schema}.host_status_updates_csv_silver');

display(bronze_df)

display(silver_df)

## 13. Iterating and Redeploying

- Edit configs, then redeploy
- Re-run onboarding after metadata changes
- Re-run the pipeline (incremental -- only processes new files)
- Full refresh (reprocesses everything)


## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `Error: no value assigned to required variable sdp_meta_schema.` | No variable override found in  ../.databricks/bundle/dev/variable-overrides.json | Run Cell 6 in Step 3 to create and set the bundle variable overrides |
| `sdp_meta_dependency is still set to __SET_ME__` | Wheel variable not configured | Set `sdp_meta_dependency` in variables or overrides |
| `Table or view not found: bronze_dataflowspec` | Onboarding not run | Run `databricks bundle run onboarding` |
| Pipeline starts but produces no tables | No matching dataflowspec rows | Check `data_flow_group` matches the pipeline config |
| `Cannot read file from workspace path` on serverless | conf files not staged to Volume | Run onboarding (the `stage_conf` task handles this) |
| Schema mismatch / `_rescued_data` not empty | Source file schema changed | Check `conf/schemas/` DDL and `cloudFiles.inferColumnTypes` setting |

#PyPi install is not working for me


You can create the wheel file from the sdp_meta repository and upload it to a volume path.
Set the parameters below to your volume path in the variable-overrides.json file and run the deploy.

```json
  "wheel_source":  f"volume_path",
  "sdp_meta_dependency": f"/Volumes/sdp_meta_lab/sdp_meta_{username}/sdp_meta/wheels/databricks_labs_sdp_meta-0.1.0-py3-none-any.whl",
```

## Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│  databricks bundle deploy                               │
├─────────────┬───────────────────────┬───────────────────┤
│ Onboarding  │  Lakeflow Declarative │  Pipeline Runner  │
│ Job         │  Pipeline             │  Job              │
│             │                       │                   │
│ stage_conf  │  init_sdp_meta_       │  Triggers the     │
│   → Volume  │  pipeline notebook    │  pipeline via     │
│ onboard     │    ↓                  │  pipeline_task    │
│   → dfspec  │  DataflowPipeline     │                   │
│     tables  │  .invoke_dlt_pipeline │                   │
│             │    ↓                  │                   │
│             │  Bronze (Autoloader)  │                   │
│             │    ↓                  │                   │
│             │  Silver (Transform)   │                   │
└─────────────┴───────────────────────┴───────────────────┘
```

## Next Steps

* <a href="$LAB 2 - Pipeline Configurations">Pipeline Configurations</a> -- hands-on exercises to extend this pipeline
* [dlt-meta Official Docs](https://databrickslabs.github.io/dlt-meta/) -- full framework reference
* [Lakeflow Declarative Pipelines](https://docs.databricks.com/en/delta-live-tables/index.html) -- platform documentation